In [1]:
import redis.asyncio as redis  
import asyncio
import nest_asyncio
nest_asyncio.apply()

from datetime import datetime

from alpaca.data.live.stock import StockDataStream
import os 

stock_stream = StockDataStream(os.environ['API_KEY'], os.environ['SECRET_KEY'])
tickers = ['AAPL','AMD','CSCO','GOOG','INTC','JNPR','META','MSFT','NFLX','NVDA','TSLA']

redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

async def push_ohlc_data(bar):
    bar = {k: v for k, v in bar}
    bar['timestamp'] = bar['timestamp'].isoformat()
    # Add the new OHLC tick to the Redis Stream
    await redis_client.xadd(f"alpaca_{bar['symbol']}", bar)

    # Trim the stream to keep only the last 30 ticks
    await redis_client.xtrim(f"ohlc_stream:{bar['symbol']}", maxlen=100)
    print(f"Pushed OHLC Tick: {bar}")


In [ ]:
stock_stream.subscribe_bars(push_ohlc_data, *tickers)
stock_stream.run()

Pushed OHLC Tick: {'symbol': 'MSFT', 'timestamp': '2025-03-20T13:48:00+00:00', 'open': 386.335, 'high': 386.64, 'low': 386.335, 'close': 386.64, 'volume': 663.0, 'trade_count': 12.0, 'vwap': 386.491475}
Pushed OHLC Tick: {'symbol': 'INTC', 'timestamp': '2025-03-20T13:48:00+00:00', 'open': 24.17, 'high': 24.17, 'low': 24.12, 'close': 24.13, 'volume': 6837.0, 'trade_count': 58.0, 'vwap': 24.131473}
Pushed OHLC Tick: {'symbol': 'NVDA', 'timestamp': '2025-03-20T13:48:00+00:00', 'open': 118.75, 'high': 118.93, 'low': 118.655, 'close': 118.81, 'volume': 15628.0, 'trade_count': 163.0, 'vwap': 118.780792}
Pushed OHLC Tick: {'symbol': 'GOOG', 'timestamp': '2025-03-20T13:48:00+00:00', 'open': 165.34, 'high': 165.35, 'low': 165.2, 'close': 165.28, 'volume': 1844.0, 'trade_count': 31.0, 'vwap': 165.305313}
Pushed OHLC Tick: {'symbol': 'CSCO', 'timestamp': '2025-03-20T13:48:00+00:00', 'open': 60.62, 'high': 60.62, 'low': 60.59, 'close': 60.615, 'volume': 1834.0, 'trade_count': 23.0, 'vwap': 60.6143